# 02 - Tag Compliance Scanner

Checks all tables have required governance tags. Reports gaps and auto-tags where possible.

**Required tags**: `owner`, `domain`, `quality_tier`
**Column-level**: `pii`, `sensitivity` (when PII detected)

In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import (
    require_widget, uc_list_tables, uc_list_schemas,
    tables_to_spark, build_exempt_schemas,
    load_exemptions, is_exempt,
)
from lib.policy import load_policy, required_table_tags as _policy_req_tags
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("control_schema", "uc_hygiene")
dbutils.widgets.text("target_catalogs", "")
dbutils.widgets.text("required_table_tags", "owner,domain,quality_tier")


catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
target_catalogs = [
    c.strip() for c in require_widget(dbutils, "target_catalogs").split(",") if c.strip()
]
_widget_tags = dbutils.widgets.get("required_table_tags").strip()
_policy      = load_policy(spark, catalog, control_schema)
required_tags = _policy_req_tags(_policy, widget_override=_widget_tags)
control_fqn = f"{catalog}.{control_schema}"

print(f"Control schema:      {control_fqn}")
print(f"Target catalogs:     {target_catalogs}")
print(f"Required table tags: {required_tags}")
import time as _t; _task_start = _t.time()

In [0]:
# Function definitions moved to src/lib/common.py — imported in widget cell above.
from databricks.sdk import WorkspaceClient

_sdk = WorkspaceClient()


print("✅ lib.common loaded; SDK client ready.")


In [0]:
from datetime import date
import uuid

scan_id   = str(uuid.uuid4())
scan_date = date.today()

# Get all tables via UC SDK
_table_rows = uc_list_tables(_sdk, target_catalogs, control_schema)
_exemptions = load_exemptions(spark, catalog, control_schema)
_table_rows  = [r for r in _table_rows if not is_exempt(r["catalog_name"], r["schema_name"], r["table_name"], _exemptions)]
all_tables  = tables_to_spark(spark, _table_rows)
all_tables.createOrReplaceTempView("all_tables_for_tags")
print(f"Tables to check across {len(target_catalogs)} catalog(s): {len(_table_rows)}")


In [0]:
# Get current tags across all target catalogs
tag_parts = []
for tc in target_catalogs:
    tag_parts.append(f"""
    SELECT catalog_name, schema_name, table_name, tag_name, tag_value
    FROM {tc}.information_schema.table_tags
    WHERE catalog_name = '{tc}'
    """)

current_tags = spark.sql("\nUNION ALL\n".join(tag_parts))
current_tags.createOrReplaceTempView("current_tags")

# Pivot: which required tags does each table have?
tag_coverage = spark.sql(f"""
SELECT
  t.table_catalog AS catalog_name,
  t.table_schema AS schema_name,
  t.table_name,
  {', '.join([f"MAX(CASE WHEN ct.tag_name = '{tag}' THEN ct.tag_value END) AS tag_{tag}" for tag in required_tags])}
FROM all_tables_for_tags t
LEFT JOIN current_tags ct
  ON t.table_catalog = ct.catalog_name
  AND t.table_schema = ct.schema_name
  AND t.table_name = ct.table_name
GROUP BY t.table_catalog, t.table_schema, t.table_name
""")
tag_coverage.createOrReplaceTempView("tag_coverage")
print("Tag coverage matrix built.")


In [0]:
# Find tables missing required tags
missing_tags_dfs = []
for tag in required_tags:
    missing = spark.sql(f"""
    SELECT 
      catalog_name, schema_name, table_name,
      '{tag}' AS missing_tag
    FROM tag_coverage
    WHERE tag_{tag} IS NULL
    """)
    missing_tags_dfs.append(missing)

from functools import reduce
from pyspark.sql import DataFrame

if missing_tags_dfs:
    all_missing = reduce(DataFrame.unionAll, missing_tags_dfs)
    all_missing.createOrReplaceTempView("missing_tags")
    missing_count = all_missing.count()
    print(f"🚨 Missing tag assignments: {missing_count}")
    
    # Show summary by tag
    all_missing.groupBy("missing_tag").count().show()
else:
    missing_count = 0
    print("✅ All tables fully tagged!")

In [0]:
# Write findings to control table
if missing_count > 0:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.scan_results
    SELECT
      '{scan_id}' AS scan_id,
      CURRENT_DATE() AS scan_date,
      'tag_compliance' AS scan_type,
      'table'         AS asset_type,
      m.catalog_name,
      m.schema_name,
      m.table_name,
      NULL AS column_name,
      'missing_required_tag' AS finding_type,
      'warning' AS finding_severity,
      CONCAT('Missing required tag: ', m.missing_tag) AS finding_detail,
      CONCAT('Add tag: SET TAG ON TABLE ', m.catalog_name, '.', m.schema_name, '.', m.table_name, ' ', m.missing_tag, ' = <value>') AS recommended_action,
      ot.tag_value AS owner_email,
      NULL AS resolved_at,
      NULL AS resolved_by
    FROM missing_tags m
    LEFT JOIN current_tags ot
      ON m.catalog_name = ot.catalog_name
      AND m.schema_name = ot.schema_name
      AND m.table_name = ot.table_name
      AND ot.tag_name = 'owner'
    """)

total_tables = len(_table_rows)
print(f"""
{'='*52}
  TAG COMPLIANCE SCAN COMPLETE
{'='*52}
  Scan ID:        {scan_id}
  Tables checked: {total_tables}
  Missing tags:   {missing_count}
  Required tags:  {required_tags}
  Results → {catalog}.{control_schema}.scan_results
{'='*52}
""")

import time as _t
try:
    spark.sql(f"""
    INSERT INTO {catalog}.{control_schema}.job_run_history VALUES (
      CURRENT_DATE(),
      'uc_hygiene_daily_governance',
      'p2_tag_compliance',
      'p2_detection',
      'success',
      {total_tables},
      {missing_count},
      {missing_count},
      int(_t.time() - _task_start),
      'required_tags={len(required_tags)} catalogs={len(target_catalogs)}',
      CURRENT_TIMESTAMP()
    )
    """)
except Exception as _e:
    print(f"Warning: could not write to job_run_history: {_e}")